# Debt payoff planner

Models single-debt amortization and multi-debt **avalanche** payoff (highest APR first).

1. Copy `config/debts.example.yaml` → `config/debts.yaml` and enter your balances.
2. Run all cells.

`debts.yaml` is git-ignored — your numbers stay local.

In [ ]:
import sys
from pathlib import Path

import yaml
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import payoff, compare, avalanche, payment_for_months

cfg_path = ROOT / "config" / "debts.yaml"
if not cfg_path.exists():
    cfg_path = ROOT / "config" / "debts.example.yaml"
    print("Using example debts — copy config/debts.example.yaml to config/debts.yaml for your numbers.")

cfg = yaml.safe_load(cfg_path.read_text())
debts = cfg["debts"]
budget = cfg["monthly_budget"]
total = sum(d["balance"] for d in debts)
print(f"Total debt: ${total:,.2f}  |  monthly budget: ${budget:,.2f}")
pd.DataFrame(debts)

## Single-debt scenarios (each debt in isolation)

In [ ]:
for d in debts:
    print(f"\n=== {d['name']}: ${d['balance']:,.2f} @ {d['apr']}% ===")
    floor = d["balance"] * d["apr"] / 100 / 12
    print(f"Interest-only floor: ${floor:.2f}/mo")
    print(compare(d["balance"], d["apr"], [floor * 1.5, budget, budget * 1.5, budget * 2]).round(2))

## Avalanche — all debts, highest APR first

In [ ]:
result = avalanche(debts, budget)
if result["months"] is None:
    print("Budget too low — doesn't cover total monthly interest.")
else:
    print(f"Debt-free in {result['months']} months (~{result['years']} years)")
    print(f"Total interest: ${result['total_interest']:,.2f}")
    for name, month in result["cleared_month"].items():
        print(f"  {name} cleared in month {month}")

In [ ]:
# Compare monthly budgets side by side.
candidates = [budget * x for x in (0.5, 0.75, 1.0, 1.25, 1.5, 2.0)]
rows = []
for b in candidates:
    r = avalanche(debts, b)
    if r["months"]:
        rows.append({"budget/mo": b, "months": r["months"], "interest": r["total_interest"]})
pd.DataFrame(rows).set_index("budget/mo").round(2)

In [ ]:
# Simulate total balance over time at your chosen budget.
bal = {d["name"]: float(d["balance"]) for d in debts}
rate = {d["name"]: d["apr"] / 100 / 12 for d in debts}
order = sorted(bal, key=lambda n: rate[n], reverse=True)
history = [sum(bal.values())]
while sum(bal.values()) > 0.005 and len(history) < 120:
    for n in order:
        bal[n] += bal[n] * rate[n]
    rem = budget
    for n in order:
        if bal[n] <= 0: continue
        p = min(rem, bal[n]); bal[n] -= p; rem -= p
        if rem <= 0: break
    history.append(max(sum(bal.values()), 0))

ax = pd.Series(history).plot(figsize=(10, 4))
ax.set_title(f"Total debt balance at ${budget:,.0f}/mo (avalanche)")
ax.set_xlabel("months"); ax.set_ylabel("$"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()